### Source Tables:
- `_exponent._bronze_epic_clarity.order_proc` — procedure orders (258M completed, DESCRIPTION only, no CPT/PROC_CODE)
- `_exponent._bronze_epic_clarity.or_log` — surgical log (73K records, SURGERY_DATE)

### Strategy:
- **order_proc**: Filter ORDER_STATUS_C = 5 (Completed), exclude labs (ORDER_TYPE_C = 7 is typical for labs)
- **or_log**: All non-voided surgical records (STATUS_C != 10)
- procedure_concept_id = 0 (no CPT codes available — backfill when codes are populated)
- procedure_type_concept_id = 32817 (EHR)
- procedure_date = COALESCE(PROC_DATE, ORDERING_DATE) for order_proc, SURGERY_DATE for or_log
- DESCRIPTION used as procedure_source_value

### Notes:
- This notebook depends on source_to_person being populated for epic_clarity
- CPT_CODE and PROC_CODE are completely unpopulated in order_proc
- provider_id, visit_occurrence_id, visit_detail_id will be NULL until those mappings exist
- Silver and gold tables already exist
- ORDER_TYPE_C 7 = Labs (203M) — excluded to avoid double-counting with measurement table

# Transformation

In [0]:
%sql
-- Create silver_procedure_occurrence temp view for Epic Clarity
CREATE OR REPLACE TEMPORARY VIEW silver_procedure_occurrence AS

-- Procedure Orders (completed, non-lab)
SELECT
  0 AS procedure_concept_id,  -- No CPT codes available
  DATE(COALESCE(op.PROC_DATE, op.ORDERING_DATE)) AS procedure_date,
  COALESCE(op.PROC_DATE, op.ORDERING_DATE) AS procedure_datetime,
  32817 AS procedure_type_concept_id,  -- EHR
  0 AS modifier_concept_id,
  CAST(op.QUANTITY AS INT) AS quantity,
  op.DESCRIPTION AS procedure_source_value,
  0 AS procedure_source_concept_id,
  NULL AS modifier_source_value,
  CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', op.PAT_ID) AS person_source_value,
  CASE
    WHEN op.AUTHRZING_PROV_ID IS NOT NULL
    THEN CONCAT_WS(CHR(31), 'epic_clarity', 'clarity_ser', 'PROV_ID', op.AUTHRZING_PROV_ID)
    ELSE NULL
  END AS provider_source_value,
  CASE
    WHEN op.PAT_ENC_CSN_ID IS NOT NULL
    THEN CONCAT_WS(CHR(31), 'epic_clarity', 'pat_enc', 'PAT_ENC_CSN_ID', CAST(op.PAT_ENC_CSN_ID AS STRING))
    ELSE NULL
  END AS visit_occurrence_source_value,
  NULL AS visit_detail_source_value,
  CONCAT_WS(CHR(31), 'epic_clarity', 'order_proc', 'ORDER_PROC_ID', CAST(op.ORDER_PROC_ID AS STRING)) AS procedure_occurrence_source_value,
  'epic_clarity' AS source_system
FROM _exponent._bronze_epic_clarity.order_proc op
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', op.PAT_ID)
  AND stp.active_flag = TRUE
WHERE op.ORDER_STATUS_C = 5  -- Completed
  AND op.ORDER_TYPE_C != 7   -- Exclude labs
  AND op.PAT_ID IS NOT NULL
  AND COALESCE(op.PROC_DATE, op.ORDERING_DATE) IS NOT NULL

UNION ALL

-- Surgical Log
SELECT
  0 AS procedure_concept_id,
  DATE(ol.SURGERY_DATE) AS procedure_date,
  ol.SURGERY_DATE AS procedure_datetime,
  32817 AS procedure_type_concept_id,
  0 AS modifier_concept_id,
  NULL AS quantity,
  COALESCE(ol.LOG_NAME, 'Surgical Procedure') AS procedure_source_value,
  0 AS procedure_source_concept_id,
  NULL AS modifier_source_value,
  CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', ol.PAT_ID) AS person_source_value,
  CASE
    WHEN ol.PRIMARY_PHYS_ID IS NOT NULL
    THEN CONCAT_WS(CHR(31), 'epic_clarity', 'clarity_ser', 'PROV_ID', ol.PRIMARY_PHYS_ID)
    ELSE NULL
  END AS provider_source_value,
  NULL AS visit_occurrence_source_value,
  NULL AS visit_detail_source_value,
  CONCAT_WS(CHR(31), 'epic_clarity', 'or_log', 'LOG_ID', CAST(ol.LOG_ID AS STRING)) AS procedure_occurrence_source_value,
  'epic_clarity' AS source_system
FROM _exponent._bronze_epic_clarity.or_log ol
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', ol.PAT_ID)
  AND stp.active_flag = TRUE
WHERE ol.PAT_ID IS NOT NULL
  AND ol.SURGERY_DATE IS NOT NULL
  AND (ol.STATUS_C IS NULL OR ol.STATUS_C != 10)  -- Exclude voided

In [0]:
# %sql
# -- Preview
# SELECT * FROM silver_procedure_occurrence LIMIT 10

In [0]:
# %sql
# -- Check for duplicates on merge key
# SELECT procedure_occurrence_source_value, COUNT(*) AS cnt
# FROM silver_procedure_occurrence
# GROUP BY procedure_occurrence_source_value
# HAVING COUNT(*) > 1
# LIMIT 10

# Write to Silver

In [0]:
%sql
MERGE INTO _exponent.omop_silver.procedure_occurrence AS t
USING (
  SELECT * FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY procedure_occurrence_source_value
        ORDER BY procedure_date DESC
      ) AS rn
    FROM silver_procedure_occurrence
  ) WHERE rn = 1
) AS s
ON t.procedure_occurrence_source_value = s.procedure_occurrence_source_value

WHEN MATCHED AND (
     NOT (t.procedure_concept_id <=> s.procedure_concept_id)
  OR NOT (t.procedure_date <=> s.procedure_date)
  OR NOT (t.procedure_datetime <=> s.procedure_datetime)
  OR NOT (t.procedure_type_concept_id <=> s.procedure_type_concept_id)
  OR NOT (t.procedure_source_value <=> s.procedure_source_value)
  OR NOT (t.procedure_source_concept_id <=> s.procedure_source_concept_id)
  OR NOT (t.modifier_source_value <=> s.modifier_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.procedure_concept_id          = s.procedure_concept_id,
  t.procedure_date                = s.procedure_date,
  t.procedure_datetime            = s.procedure_datetime,
  t.procedure_type_concept_id     = s.procedure_type_concept_id,
  t.modifier_concept_id           = s.modifier_concept_id,
  t.quantity                       = s.quantity,
  t.procedure_source_value        = s.procedure_source_value,
  t.procedure_source_concept_id   = s.procedure_source_concept_id,
  t.modifier_source_value         = s.modifier_source_value,
  t.source_system                 = s.source_system,
  t.last_mod_tsp                  = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  person_source_value,
  provider_source_value,
  visit_occurrence_source_value,
  visit_detail_source_value,
  procedure_occurrence_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.procedure_concept_id,
  s.procedure_date,
  s.procedure_datetime,
  s.procedure_type_concept_id,
  s.modifier_concept_id,
  s.quantity,
  s.procedure_source_value,
  s.procedure_source_concept_id,
  s.modifier_source_value,
  s.person_source_value,
  s.provider_source_value,
  s.visit_occurrence_source_value,
  s.visit_detail_source_value,
  s.procedure_occurrence_source_value,
  s.source_system,
  current_timestamp()
);

In [0]:
# %sql
# -- Verify silver
# SELECT * FROM _exponent.omop_silver.procedure_occurrence
# WHERE source_system = 'epic_clarity'
# LIMIT 10

# Register Procedure Occurrence IDs

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_procedure_occurrence (
    source_system,
    procedure_occurrence_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.procedure_occurrence_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, procedure_occurrence_source_value, last_mod_tsp
    FROM _exponent.omop_silver.procedure_occurrence
    WHERE source_system = 'epic_clarity'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_procedure_occurrence x
  ON s.procedure_occurrence_source_value = x.procedure_occurrence_source_value;

# Write to Gold

In [0]:
%sql
-- Merge to Gold layer
-- MERGE INTO _exponent.omop.procedure_occurrence AS gold
MERGE INTO _exponent.omop_epic.procedure_occurrence AS gold
USING (
  SELECT
    spo.procedure_occurrence_id,
    stp.person_id,
    s.procedure_concept_id,
    s.procedure_date,
    s.procedure_datetime,
    s.procedure_type_concept_id,
    s.modifier_concept_id,
    s.quantity,
    NULL AS provider_id,
    NULL AS visit_occurrence_id,
    NULL AS visit_detail_id,
    s.procedure_source_value,
    s.procedure_source_concept_id,
    s.modifier_source_value
  FROM _exponent.omop_silver.procedure_occurrence s
  JOIN _exponent.omop_mapping.source_to_procedure_occurrence spo
    ON spo.procedure_occurrence_source_value = s.procedure_occurrence_source_value
   AND spo.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.active_flag = TRUE
  WHERE s.source_system = 'epic_clarity'
) AS src
ON gold.procedure_occurrence_id = src.procedure_occurrence_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                      = src.person_id,
  gold.procedure_concept_id           = src.procedure_concept_id,
  gold.procedure_date                 = src.procedure_date,
  gold.procedure_datetime             = src.procedure_datetime,
  gold.procedure_type_concept_id      = src.procedure_type_concept_id,
  gold.modifier_concept_id            = src.modifier_concept_id,
  gold.quantity                        = src.quantity,
  gold.provider_id                    = src.provider_id,
  gold.visit_occurrence_id            = src.visit_occurrence_id,
  gold.visit_detail_id                = src.visit_detail_id,
  gold.procedure_source_value         = src.procedure_source_value,
  gold.procedure_source_concept_id    = src.procedure_source_concept_id,
  gold.modifier_source_value          = src.modifier_source_value

WHEN NOT MATCHED THEN INSERT (
  procedure_occurrence_id,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value
)
VALUES (
  src.procedure_occurrence_id,
  src.person_id,
  src.procedure_concept_id,
  src.procedure_date,
  src.procedure_datetime,
  src.procedure_type_concept_id,
  src.modifier_concept_id,
  src.quantity,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.procedure_source_value,
  src.procedure_source_concept_id,
  src.modifier_source_value
);

# Validation

In [0]:
# %sql
# -- Layer counts
# SELECT 'Silver' AS layer, COUNT(*) AS record_count FROM _exponent.omop_silver.procedure_occurrence WHERE source_system = 'epic_clarity'
# UNION ALL
# SELECT 'Mapping' AS layer, COUNT(*) AS record_count FROM _exponent.omop_mapping.source_to_procedure_occurrence WHERE source_system = 'epic_clarity'
# UNION ALL
# SELECT 'Gold' AS layer, COUNT(*) AS record_count FROM _exponent.omop.procedure_occurrence

In [0]:
# %sql
# -- Verify gold
# SELECT * FROM _exponent.omop.procedure_occurrence
# LIMIT 10

In [0]:
# %sql
# -- Top procedure descriptions
# SELECT procedure_source_value, COUNT(*) AS cnt
# FROM _exponent.omop.procedure_occurrence
# GROUP BY procedure_source_value
# ORDER BY cnt DESC
# LIMIT 20